# 03 Preprocessing

Turn the register's text into a rule a program can evaluate at a given time, and say so when it
cannot. The parsers **fail closed**: they must account for every character of a field, and
anything they cannot read becomes an issue with a reason, never a partial answer.

*Runs offline on the committed snapshot, which keeps the raw fields beside the parsed ones.
Section 6 reads git history, so it needs a full clone.*

In [1]:
import json
import os
import subprocess
import sys
import types
from datetime import date
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline").is_dir()))
sys.path[:0] = ["pipeline", "analysis"]

import geopandas as gpd
import holidays
import pandas as pd
from IPython.display import Markdown

import export_web
import process
import report_stats as rs
import rules

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)
areas = gpd.read_parquet(rs.SOURCE)

## 1. What goes in

The size of the problem is how many ways each field is written:

In [2]:
Markdown(rs.raw_spellings(areas))

### Distinct raw spellings per field

| Field                        |   Distinct spellings |   Missing or blank |
|:-----------------------------|---------------------:|-------------------:|
| Hours (`voimassaolo`)        |                   66 |               2922 |
| Time limit (`kesto`)         |                   25 |               6247 |
| Season (`kausi`)             |                   15 |               8274 |
| Space type (`tyyppi`)        |                   25 |               6866 |
| Free-text note (`lisatieto`) |                  201 |               6993 |


For each field: list every spelling, decide what it means, and parse all of them, so a spelling
never seen before is flagged.

## 2. Rule type

The class `luokka` gives the kind of rule, and a special space type `tyyppi` overrides it:

In [3]:
per_class = areas["luokka"].value_counts()
pd.DataFrame(
    [(k, per_class.get(k, 0), rule, f"{limit} min" if limit else "none", name)
     for k, (rule, limit, name) in rules.LUOKKA_RULES.items()],
    columns=["class", "areas", "rule type", "limit the class states", "meaning"],
).set_index("class")

,areas,rule type,limit the class states,meaning
class,,,,
1,406,free_limited,none,Free short-term parking
2,34,free_limited,none,Free long-term parking
3,30,paid,60 min,"Single payment, up to 1 hour"
4,83,paid,120 min,"Single payment, up to 2 hours"
5,522,paid,240 min,"Single payment, up to 4 hours"
6,3089,paid,none,Paid without a resident or business permit; hours vary by location
7,429,paid,60 min,Single payment up to 1 hour without a resident or business permit
8,1599,free_limited,none,Free short-term parking without a permit; use a parking disc
9,283,banned_hours,none,Parking allowed outside the no-parking hours


Eleven classes, five rule types: the app needs the kind of rule, and each class's detail is kept
elsewhere or dropped knowingly. The class limit becomes `duration_min` when an area states none.
The permit exemption of classes 6 and 7 is dropped, which is right for a visitor and strict for a
resident. The zone rate of class 10 is dropped, since the register holds no prices.

The **parking disc** is not a class detail. Free parking under a time limit of less than 24 hours
needs a disc, set to the next half or full hour after arrival. Class 8 says so; class 1 needs it
too:

In [4]:
disc = rs.needs_disc(areas)
print(f"{disc.sum():,} areas need a disc")
areas.loc[disc, "luokka"].value_counts().rename("areas").rename_axis("class").to_frame()

1,650 areas need a disc


,areas
class,
8.0,1587
1.0,63


The app answered these *free parking, 4 hours max* without mentioning the disc; PR #44 fixes it.

Class 0, or no class at all, holds both bans and reserved bays, and only the space type tells
them apart:

In [5]:
pd.crosstab(areas["luokka"].fillna(0).astype(int).rename("class"), areas["rule_type"], margins=True, margins_name="all")

rule_type,always_banned,banned_hours,free_limited,paid,reserved,unknown,all
class,,,,,,,
0,884,0,0,0,980,62,1926
1,0,0,406,0,0,0,406
2,0,0,31,0,3,0,34
3,0,0,0,30,0,0,30
4,0,0,0,83,0,0,83
5,0,0,0,522,0,0,522
6,0,0,0,3089,0,0,3089
7,0,0,0,429,0,0,429
8,1,0,1598,0,0,0,1599


In [6]:
def outcome(klass, space_type):
    rule, limit, issue = rules.rule_type(klass, space_type)
    return {"class": "none" if klass is None else klass, "space type": space_type or "none",
            "rule type": rule, "issue": f"{issue[0]}: {issue[1]}" if issue else ""}

pd.DataFrame([outcome(k, t) for k, t in [(6, None), (6, "Taxi"), (0, "Pysäköintikielto"),
                                          (None, "Sähköpotkulauta"), (6, "Uusi merkki")]])

,class,space type,rule type,issue
0,6,none,paid,
1,6,Taxi,reserved,
2,0,Pysäköintikielto,always_banned,
3,none,Sähköpotkulauta,reserved,
4,6,Uusi merkki,unknown,unreadable: unknown space type 'Uusi merkki'


An unknown space type is flagged rather than passed to the class: falling back would make a new
kind of reserved bay "paid".

## 3. Hours

A bare range is Monday to Friday, a bracketed one Saturday, a third one Sunday: `9-21, (9-18)` is
weekdays 9-21 and Saturday 9-18.

In [7]:
for raw in ["9-21", "9-21, (9-18)", "8-18, (8-14), 10-14", "9.21, (9-18)", "7-9, 15-17", "8-18 la"]:
    print(f"{raw!r:24} -> {rules.parse_hours(raw)}")

'9-21'                   -> ({'mon_fri': [9, 21]}, None)
'9-21, (9-18)'           -> ({'mon_fri': [9, 21], 'sat': [9, 18]}, None)
'8-18, (8-14), 10-14'    -> ({'mon_fri': [8, 18], 'sat': [8, 14], 'sun': [10, 14]}, None)
'9.21, (9-18)'           -> ({'mon_fri': [9, 21], 'sat': [9, 18]}, None)
'7-9, 15-17'             -> (None, ('ambiguous', "two ranges with no brackets to tell the days apart: '7-9, 15-17'"))
'8-18 la'                -> (None, ('unreadable', "the hours '8-18 la' also say 'la', which we cannot read"))


`9.21, (9-18)` is a real typo, accepted. `8-18 la` is invented. `7-9, 15-17` is real: two bare
ranges are a split weekday window, not ambiguous as the parser says, but the hours hold one range
per day, so it is flagged (board task 3.6). Every spelling in the register, where a blank
*read as* means flagged:

In [8]:
def every_spelling(field, parser, show):
    """Every distinct spelling of a field, the areas using it, and what the parser made of it."""
    values = areas.loc[rs.stated(areas[field]), field]
    rows = []
    for raw, n in values.value_counts().items():
        value, issue = parser(raw)
        rows.append({"raw": raw, "areas": n, "read as": "" if issue else show(value),
                     "issue": issue[0] if issue else ""})
    print(f"{values.nunique()} spellings; {len(areas) - len(values):,} areas leave the field blank")
    return pd.DataFrame(rows).set_index("raw")

every_spelling("voimassaolo", rules.parse_hours, lambda hours: rs.readable(json.dumps(hours)))

66 spellings; 2,922 areas leave the field blank


,areas,read as,issue
raw,,,
"9-21, (9-18)",1907,"mon_fri 9-21, sat 9-18",
9-21,1708,mon_fri 9-21,
8-20,1487,mon_fri 8-20,
8-17,151,mon_fri 8-17,
"9-21, (9-21), 9-21",114,"mon_fri 9-21, sat 9-21, sun 9-21",
8-18,92,mon_fri 8-18,
"7-18, (9-15)",36,"mon_fri 7-18, sat 9-15",
7-18,30,mon_fri 7-18,
9-21\n,30,mon_fri 9-21,


## 4. Time limits

In minutes: `0` means the register says there is no limit, absent means none is stated. A Russian
duplicate after a comma is accepted; anything else is flagged, such as `max 60 min lauantai`,
which hides a day ("Saturday") inside the limit.

In [9]:
every_spelling("kesto", rules.parse_duration, rs.duration_label)

25 spellings; 6,247 areas leave the field blank


,areas,read as,issue
raw,,,
4 h,1794,4 h,
60 min,473,1 h,
2 h,105,2 h,
24 h,22,24 h,
4h,16,4 h,
30 min,15,30 min,
"30 min, (30 мин)",14,30 min,
"60 min, (60 мин)",12,1 h,
max 60 min lauantai,8,,unreadable


Where the class and the field disagree, the area is flagged rather than one being chosen:

In [10]:
areas.loc[areas["reason"].str.contains("the class allows", na=False), ["id", "luokka", "kesto", "reason"]]

,id,luokka,kesto,reason
1326,9385,5.0,24 h,the class allows 240 min but the register says 1440 min
1427,9487,5.0,24 h,the class allows 240 min but the register says 1440 min
1565,9626,3.0,30min,the class allows 60 min but the register says 30 min
1590,9651,5.0,24 h,the class allows 240 min but the register says 1440 min
1744,9810,5.0,2 h,the class allows 240 min but the register says 120 min
2141,10211,5.0,24 h,the class allows 240 min but the register says 1440 min
3268,11361,5.0,24 h,the class allows 240 min but the register says 1440 min
4306,12428,5.0,24 h,the class allows 240 min but the register says 1440 min
8286,16582,4.0,4,the class allows 120 min but the register says 240 min
8287,16583,4.0,4,the class allows 120 min but the register says 240 min


## 5. Seasons and notes

In [11]:
every_spelling("kausi", rules.parse_season, lambda season: "all year" if season is None else rs.readable(json.dumps(season)))

15 spellings; 8,274 areas leave the field blank


,areas,read as,issue
raw,,,
0,410,all year,
(1.1.-31.12),27,all year,
(1.1.-31.12.),11,all year,
1.4. - 31.10.,10,1.4.-31.10.,
(1.4.-31.10.),7,1.4.-31.10.,
Ympärivuotinen,4,all year,
(1.5. - 31.10.),2,1.5.-31.10.,
Vain sataman luvalla,2,,unreadable
(1.11.-31.3.),1,1.11.-31.3.,


`lisatieto` is free text. A sign reference number is kept as metadata; anything else is a
condition we do not parse, shown to the driver verbatim, and the area is flagged:

In [12]:
conditions = areas.loc[areas["issue_codes"].str.contains("note", na=False), "lisatieto"]
print(f"{len(conditions)} areas with a condition we do not parse")
conditions.value_counts().head(8)

301 areas with a condition we do not parse


lisatieto
Moottorikäyttöisillä ajoneuvoilla ajo kielletty 22-6 (22-6) 22-6, Ei koske huoltoajoa, tontilleajoa, asukaspysäköintitunnuksella F, Gäller ej servicekörning, tomtinfart, med boendeparkeringstecknet F    66
C41, E8                                                                                                                                                                                                    41
reunakiven päällä, 6909                                                                                                                                                                                    29
7539, C39                                                                                                                                                                                                  16
C41                                                                                                                                                                   

Some of those are only sign codes with a letter, such as `C41, E8`, which `parse_extra` does not
take as a reference:

In [13]:
lettered = rs.lettered_sign_codes(areas)
print(f"{lettered.sum()} areas flagged for that alone")
areas.loc[lettered, "rule_type"].value_counts()

116 areas flagged for that alone


rule_type
reserved         89
always_banned    27
Name: count, dtype: int64

All are reserved bays or no-parking areas, whose rule a sign reference cannot change. That is
safe, but probably too safe; it is an open question on the board.

## 6. Failing closed, and what it changed

Our first parser read what it recognised and dropped the rest. Loaded from git history and run
over every spelling in the register:

In [14]:
def parser_at(commit):
    """pipeline/rules.py as it was at a commit."""
    module = types.ModuleType(f"rules_{commit}")
    exec(subprocess.run(["git", "show", f"{commit}:pipeline/rules.py"],
                        capture_output=True, text=True, check=True).stdout, module.__dict__)
    return module

first = parser_at("4e729f3")   # "Parse the parking register into machine-readable rules"
changed = []
for field, parser in [("voimassaolo", "parse_hours"), ("kesto", "parse_duration"),
                      ("kausi", "parse_season"), ("lisatieto", "parse_extra")]:
    for raw, n in areas.loc[rs.stated(areas[field]), field].value_counts().items():
        old_value, old_issue = getattr(first, parser)(raw)
        new_value, new_issue = getattr(rules, parser)(raw)
        if old_issue is None and new_issue is not None:
            changed.append({"field": field, "raw": raw, "areas": n, "first parser read": old_value, "now": new_issue[0]})
pd.DataFrame(changed)

,field,raw,areas,first parser read,now
0,voimassaolo,"7-15, Maksullinen (9-18)",5,"{'mon_fri': [7, 15], 'sat': [9, 18]}",unreadable
1,voimassaolo,"7-18, Maksullinen (9-18)",3,"{'mon_fri': [7, 18], 'sat': [9, 18]}",unreadable


In [15]:
areas.loc[areas["voimassaolo"].str.contains("Maksullinen", na=False), ["id", "district", "voimassaolo", "kesto", "status"]]

,id,district,voimassaolo,kesto,status
731,8778,KAMPINMALMI,"7-18, Maksullinen (9-18)",max 60 min lauantai,uncertain
873,8928,KAMPINMALMI,"7-15, Maksullinen (9-18)",max 60 min lauantai,uncertain
1152,9210,KAMPINMALMI,"7-18, Maksullinen (9-18)",max 60 min lauantai,uncertain
1233,9291,KAMPINMALMI,"7-18, Maksullinen (9-18)",max 60 min lauantai,uncertain
2282,10354,KAMPINMALMI,"7-15, Maksullinen (9-18)",max 60 min lauantai,uncertain
3712,11830,KAMPINMALMI,"7-15, Maksullinen (9-18)",max 60 min lauantai,uncertain
4921,13055,KAMPINMALMI,"7-15, Maksullinen (9-18)",max 60 min lauantai,uncertain
6615,14782,KAMPINMALMI,"7-15, Maksullinen (9-18)",max 60 min lauantai,uncertain


Two spellings, 8 areas on one stretch of street. *Maksullinen* ("paid") sits on the bracketed
Saturday range, and the limit says *lauantai*, so the week is:

| Day | Hours | Rule |
|---|---|---|
| Monday to Friday | 7-15 (5 areas) or 7-18 (3) | no parking |
| Monday to Friday | otherwise | free, no limit |
| Saturday | 9-18 | paid, 60 minutes max |
| Saturday | otherwise | free, no limit |
| Sunday and holidays | all day | free, no limit |

The first parser read Saturday as a ban. These areas were flagged anyway, but only through their
limit field: right by luck. Failing closed per field removes the luck. The app still says *check
the sign*, because an area has one rule type and here weekdays are a ban while Saturday is paid
(board task 3.6).

In [16]:
# Not in the register, but kept as a case in tests/test_status.py
print("first parser:", first.parse_hours("7-18 7-15"))
print("today:       ", rules.parse_hours("7-18 7-15"))

first parser: ({'mon_fri': [7, 18]}, None)
today:        (None, ('ambiguous', "two ranges with no brackets to tell the days apart: '7-18 7-15'"))


## 7. A status for every area

| Status | Meaning | The app says |
|---|---|---|
| `official` | a complete rule | the answer |
| `missing_hours` | the kind of rule, but not when it applies | the register does not publish hours here; for 333 areas, a labelled estimate from stage 5 |
| `uncertain` | something unreadable or contradictory | check the sign, and why |

A ban or a reserved bay applies at every hour, so it needs no hours and is not missing them. One
area, end to end:

In [17]:
row = areas.loc[areas["id"] == 8928, process.RAW_FIELDS].iloc[0].to_dict()
print("in: ", row)
print("out:", process.classify(row))

in:  {'luokka': 9.0, 'luokka_nimi': 'Pysäköinti sallittu pysäköintikieltoajan ulkopuolella', 'tyyppi': '', 'voimassaolo': '7-15, Maksullinen (9-18)', 'kesto': 'max 60 min lauantai', 'kausi': '', 'lisatieto': ''}
out: {'rule_type': 'banned_hours', 'class_name_en': 'Parking allowed outside the no-parking hours', 'space_type_en': None, 'hours': None, 'duration_min': None, 'season': None, 'extra_info': None, 'status': 'uncertain', 'issue_codes': 'unreadable', 'reason': "the hours '7-15, Maksullinen (9-18)' also say 'Maksullinen', which we cannot read; unreadable duration 'max 60 min lauantai'"}


In [18]:
pd.crosstab(areas["rule_type"], areas["status"], margins=True, margins_name="all")

status,missing_hours,official,uncertain,all
rule_type,,,,
always_banned,0,841,44,885
banned_hours,36,234,13,283
free_limited,381,1651,3,2035
paid,561,3776,109,4446
reserved,0,889,154,1043
unknown,0,0,62,62
all,978,7391,385,8754


In [19]:
Markdown(rs.issue_codes(areas))

### Issue codes

| Issue code   |   Areas |
|:-------------|--------:|
| note         |     301 |
| unreadable   |      86 |
| ambiguous    |       1 |

385 uncertain areas, 3 with more than one code.


## 8. Which window applies on a date

`rules.window_for` decides, and the app's holiday calendar is built with it
([02 data collection](02_data_collection.ipynb), §5):

In [20]:
finnish = set(holidays.Finland(years=2026))
days = [date(2026, 12, 22), date(2026, 12, 23), date(2026, 12, 25), date(2026, 12, 26), date(2026, 12, 28)]
pd.DataFrame({"weekday": [d.strftime("%A") for d in days], "window": [rules.window_for(d, finnish) for d in days]},
             index=pd.Index(days, name="date"))

,weekday,window
date,,
2026-12-22,Tuesday,mon_fri
2026-12-23,Wednesday,sat
2026-12-25,Friday,sun
2026-12-26,Saturday,sun
2026-12-28,Monday,mon_fri


## 9. Checks, outputs, and reproduction

`process.check` refuses to write repeated or missing ids, missing geometry, non-polygons, and
coordinates that are not metres around Helsinki (the trap from 02, §3). Two broken copies:

In [21]:
process.check(areas)
broken = {"a duplicated id": pd.concat([areas, areas.head(1)]),
          "coordinates in degrees": areas.to_crs(4326).set_crs(3879, allow_override=True)}
for name, copy in broken.items():
    try:
        process.check(copy)
        print(f"{name}: NOT caught")
    except ValueError as error:
        print(f"{name}: refused, {str(error)[:60]}")

a duplicated id: refused, parking area ids must be unique and present
coordinates in degrees: refused, coordinates are not metres around Helsinki: [24.84436636 60.


Two outputs come from one parsed table. **`data/processed/parking_rules.parquet`** is the analysis
snapshot: one row per area, with the raw fields beside what they became:

| Register field | Becomes |
|---|---|
| `luokka`, `luokka_nimi`, `tyyppi` | `rule_type`, `class_name_en`, `space_type_en` |
| `voimassaolo` | `hours`, e.g. `{"mon_fri": [9, 21], "sat": [9, 18]}` |
| `kesto`, or the class's limit | `duration_min` |
| `kausi` | `season` |
| `lisatieto` | `extra_info`, when it is a condition |
| all of them | `status`, `issue_codes`, `reason` |
| geometry | `district` |

Stage 5 adds two more, `predicted_hours` and `predicted_from_m`, on the `missing_hours` areas it can
estimate ([05 learning](05_learning.ipynb)).

**`web/public/data/parking_areas.geojson`** is what the app loads: only the fields it reads, the
estimates included, in GPS coordinates, with an empty field left out rather than null.

In [22]:
app = json.loads(export_web.TARGET.read_text())
next(f["properties"] for f in app["features"] if f["properties"]["status"] == "uncertain")

{'id': 8015,
 'rule_type': 'paid',
 'luokka': 6,
 'hours': {'mon_fri': [9, 21]},
 'status': 'uncertain',
 'reason': 'the sign carries a condition we do not read; check it yourself',
 'extra_info': 'Moottorikäyttöisillä ajoneuvoilla ajo kielletty 22-6 (22-6) 22-6, Ei koske huoltoajoa, tontilleajoa, asukaspysäköintitunnuksella F, Gäller ej servicekörning, tomtinfart, med boendeparkeringstecknet F'}

Does the committed data match today's code? `process.reparse` re-derives every parsed column from
the raw fields (a test runs the same check), and `export_web.py --check` rebuilds the app's file:

In [23]:
process.reparse(areas).rename("areas that differ").to_frame()

,areas that differ
rule_type,0
class_name_en,0
space_type_en,0
hours,0
duration_min,0
season,0
extra_info,0
status,0
issue_codes,0
reason,0


In [24]:
print(subprocess.run([sys.executable, "pipeline/export_web.py", "--check"], capture_output=True, text=True).stdout)

web/public/data/parking_areas.geojson matches the committed data



## 10. Reflection

**Hands on:** per area, a rule type, hours, a limit, a season, a status with its reason, a
district, and the raw fields to trace it back.

**The canvas planned** to parse the hours, normalise the limits, combine class and space type
into five rule types, match roadworks to areas, and show unusual sign text as written. All of it
was done except the roadworks.

**What did not, and why:** our first parser did not fail closed (§6).

**What changed:** a sixth rule type, `unknown`, for what cannot be classified; a status for every
area, which was not planned and became the project's central output; roadworks dropped (02).

**What we would do differently:** fail closed per field from the first parser; keep the
vocabulary in one place from the start (the English labels once drifted between a notebook and
the pipeline); list every spelling before writing a parser, not after it fails.

**Future steps:** richer hours, with split days and a rule per window (board task 3.6); lettered
sign codes as references (§5); free-text conditions read by a language model (S1); a cross-check
against Digiroad's sign panels.

**Experience and learning:** pure functions returning `(value, issue)`, regular expressions that
must match a whole field, and tests that pin every refusal. Wrangling is enumeration: every value,
then its meaning. A partial parse is worse than none, and *I cannot read this* is a valid output.

> *Each of us: one line on what you took from this stage.*